# Worked End-to-End Example

This notebook follows a single source from input spectrum through atmosphere, telescope throughput, detector response, order sorting, interferogram generation, FFT recovery, and analytical SNR estimation.

Use it as the first demonstration notebook for new users of the simulator.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

from mkid_ifts_sim import InstrumentConfig, load_template, run_full_simulation, snr_from_time
from mkid_ifts_sim.etc import prepare_observation

config = InstrumentConfig(
    n_steps=512,
    n_sigma=2048,
    delta_x_m=2.0e-6,
    t_exp_per_step_s=2.0,
    strategy="probabilistic",
    moon_phase="new",
    apodization="none",
)
source = load_template("hii_region", sigma_grid_cm=config.sigma_grid(), line_flux=2.0e-2)
observation = prepare_observation(source, config)
full_result = run_full_simulation(source, config, include_noise=True, include_sky=True)
analytical = snr_from_time(source, config, config.total_observing_time_s)

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 9), constrained_layout=True)

wavelength_nm = config.wavelength_grid_nm()[::-1]
axes[0, 0].plot(wavelength_nm, source.flux_photons_per_s_cm2_nm[::-1], label="Source")
axes[0, 0].plot(wavelength_nm, observation.sky_rate_per_nm[::-1], label="Sky at detector")
axes[0, 0].set_xlabel("Wavelength [nm]")
axes[0, 0].set_ylabel("Flux / rate")
axes[0, 0].set_title("Input source and sky")
axes[0, 0].legend()

axes[0, 1].plot(full_result.throughput.wavelength_nm[::-1], full_result.detector_rates.total_rate_per_nm[::-1])
axes[0, 1].set_xlabel("Wavelength [nm]")
axes[0, 1].set_ylabel("Detected rate [photons s$^{-1}$ nm$^{-1}$]")
axes[0, 1].set_title("Detected broadband rate")

axes[1, 0].plot(full_result.config.delta_x_m * 1.0e6 * np.arange(full_result.order_interferograms.shape[1]), full_result.order_interferograms[0])
axes[1, 0].set_xlabel("Step index scaled by delta_x [micron]")
axes[1, 0].set_ylabel("Interferogram signal")
axes[1, 0].set_title("First recovered-order interferogram")

axes[1, 1].plot(full_result.stitched_spectrum.wavelength_nm[::-1], full_result.stitched_spectrum.flux_photons_per_s_cm2_nm[::-1], label="Full simulation")
axes[1, 1].plot(analytical.wavelength_nm[::-1], analytical.snr[::-1], label="Analytical SNR")
axes[1, 1].set_xlabel("Wavelength [nm]")
axes[1, 1].set_title("Recovered spectrum and analytical SNR")
axes[1, 1].legend()

plt.show()

In [ ]:
print(f"Number of folding orders: {full_result.order_layout.n_orders}")
print(f"Saturation warning: {full_result.saturation_warning}")
print(f"Peak analytical SNR: {np.max(analytical.snr):.2f}")
print(f"Median analytical SNR: {np.median(analytical.snr):.2f}")

for idx, (lo, hi) in enumerate(full_result.order_layout.order_bounds_cm[:5]):
    mean_contamination = np.mean(full_result.source_orders.contamination_fraction[(config.sigma_grid() >= lo) & (config.sigma_grid() <= hi)])
    print(f"Order {idx}: {lo:.0f}-{hi:.0f} cm^-1, mean contamination fraction={mean_contamination:.3e}")